# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer 

fatal: destination path 'SQuAD-explorer' already exists and is not an empty directory.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp) 

In [4]:
type(train_data) 

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"]) 

list

In [7]:
len(train_data["data"]) 

442

In [8]:
type(train_data["data"][0]) 

dict

In [9]:
train_data["data"][0].keys() 

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"] 

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"]) 

55

In [12]:
train_data["data"][0]["paragraphs"][0] 

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"]) 

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [14]:
# YOUR CHANGES HERE

%pip install pandas 

import pandas as pd

rows = []

for article in train_data["data"]:
    title = article["title"]
    
    for i, paragraph in enumerate(article["paragraphs"]):
        context = paragraph["context"]
        
        rows.append({
            "document_title": title,
            "paragraph_index": i,
            "paragraph_context": context
        })

df = pd.DataFrame(rows)

df.to_csv("parsed.tsv", sep="\t", index=False) 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [15]:
# YOUR CHANGES HERE

%pip install scikit-learn

from sklearn.feature_extraction.text import HashingVectorizer

# read the parsed paragraphs from Part 2
parsed_df = pd.read_csv("parsed.tsv", sep="\t")

# build 1024-dimensional paragraph vectors
paragraph_vectorizer = HashingVectorizer(
    n_features=1024,
    stop_words="english",
    norm="l2",
    alternate_sign=True
)

paragraph_vectors = paragraph_vectorizer.fit_transform(parsed_df["paragraph_context"])

paragraph_vectors.shape 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


(18896, 1024)

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [16]:
# YOUR CHANGES HERE

import json

vector_rows = []

for i in range(paragraph_vectors.shape[0]):
    vector = paragraph_vectors[i].toarray()[0]
    vector_json = json.dumps(vector.tolist())
    
    vector_rows.append({
        "document_title": parsed_df.loc[i, "document_title"],
        "paragraph_index": parsed_df.loc[i, "paragraph_index"],
        "paragraph_vector_json": vector_json
    })

vector_df = pd.DataFrame(vector_rows)

vector_df.to_csv("paragraph-vectors.tsv.gz", sep="\t", index=False) 

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [19]:
# YOUR CHANGES HERE

questions_df = pd.read_csv("Mod8-Week10-Questions.tsv", sep="\t")

question_vectors = paragraph_vectorizer.transform(questions_df["question"])

question_vectors.shape 

(100, 1024)

Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [20]:
# YOUR CHANGES HERE

import json

question_rows = []

for i in range(question_vectors.shape[0]):
    vector = question_vectors[i].toarray()[0]
    vector_json = json.dumps(vector.tolist())
    
    question_rows.append({
        "question_id": questions_df.loc[i, "question_id"],
        "question_vector_json": vector_json
    })

question_vector_df = pd.DataFrame(question_rows)

question_vector_df.to_csv("question-vectors.tsv", sep="\t", index=False) 

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [21]:
# YOUR CHANGES HERE

from sklearn.neighbors import NearestNeighbors

# fit nearest neighbors model on paragraph vectors
nn_model = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn_model.fit(paragraph_vectors)

# find top 5 nearest paragraph vectors for each question vector
distances, indices = nn_model.kneighbors(question_vectors)

distances.shape, indices.shape 

((100, 5), (100, 5))

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [22]:
# YOUR CHANGES HERE

match_rows = []

for q_idx in range(len(indices)):
    question_id = questions_df.loc[q_idx, "question_id"]
    
    for rank, p_idx in enumerate(indices[q_idx]):
        match_rows.append({
            "question_id": question_id,
            "question_rank": rank,
            "document_title": parsed_df.loc[p_idx, "document_title"],
            "paragraph_index": parsed_df.loc[p_idx, "paragraph_index"]
        })

matches_df = pd.DataFrame(match_rows)

matches_df.to_csv("question-matches.tsv", sep="\t", index=False) 

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?  


In [26]:
questions_df.head() 

,question_id,question
0,1,What was the goal of the abuse of region project?
1,4,How many satellites in the Beidou-1 constellat...
2,7,When did Beyoncé receive ten nominations for ...
3,10,"With which goddess did Sulla, Pompey, and Juli..."
4,13,What area is considered to have a desert clima...


In [27]:
for q_idx in range(5):
    print("QUESTION:", questions_df.loc[q_idx, "question"])
    print("-----")
    
    for rank, p_idx in enumerate(indices[q_idx]):
        print(f"Rank {rank}:")
        print(parsed_df.loc[p_idx, "paragraph_context"][:200])  # show first 200 chars
        print()
        
    print("====================================\n")

QUESTION: What was the goal of the abuse of region project?
-----
Rank 0:
When optimally designed within a given core saturation constraint and for a given active current (i.e., torque current), voltage, pole-pair number, excitation frequency (i.e., synchronous speed), and 

Rank 1:
In general, Hokkien dialects have 5 to 7 phonemic tones. According to the traditional Chinese system, however, there are 7 to 9 "tones",[citation needed] more correctly termed tone classes since two o

Rank 2:
One such institution is the Centre for the Study of Ancient Documents (CSAD) founded by and located centrally at Oxford University, Great Britain. Among its many activities CSAD numbers "a long-term p

Rank 3:
In the late 1990s and early 2000s, the IFAB experimented with ways of creating a winner without requiring a penalty shootout, which was often seen as an undesirable way to end a match. These involved 

Rank 4:
Cork is home to the RTÉ Vanbrugh Quartet, and to many musical acts, including John Spi

Submit "worst-paragraphs.tsv" in Gradescope.

In [28]:
worst_rows = []

for q_idx in range(len(indices)):
    question_id = questions_df.loc[q_idx, "question_id"]
    
    # worst match = last one (rank 4)
    worst_p_idx = indices[q_idx][4]
    
    worst_rows.append({
        "question_id": question_id,
        "document_title": parsed_df.loc[worst_p_idx, "document_title"],
        "paragraph_index": parsed_df.loc[worst_p_idx, "paragraph_index"]
    })

worst_df = pd.DataFrame(worst_rows)

worst_df.to_csv("worst-paragraphs.tsv", sep="\t", index=False)

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [31]:
worst_df.columns

Index(['question_id', 'document_title', 'paragraph_index'], dtype='str')

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

I did not discuss this assignment with any classmates.

I used the scikit-learn library to implement the HashingVectorizer and NearestNeighbors models for vectorization and similarity search. I also referred to the example code notebooks provided in the homework materials for guidance on implementation.

I used ChatGPT to help clarify assignment instructions, debug code errors, and better understand concepts such as vectorization and nearest neighbor search. All code was reviewed, tested, and understood before submission.